# 03 – Modelagem: baseline recalculado, três famílias de modelos e comparação

Executa o pipeline completo (`src/executar.py`): partição temporal, validação cruzada com janela expansiva **dentro do treino**, baseline recalculado na mesma partição, quatro candidatos (Ridge, árvore, random forest e gradient boosting) e o teste avaliado uma única vez. Sem `dados/processado/base_mensal.csv`, roda em **modo demonstração** (dados sintéticos): nenhum número desse modo é resultado do projeto.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
pd.set_option("display.width", 160, "display.max_columns", 30)
from src import config
from src.executar import executar, ler_resultados
R = executar()
print(R["aviso"] or "Dados reais.")

Buscando hiperparâmetros: Linear (Ridge)


Buscando hiperparâmetros: Árvore de decisão


Buscando hiperparâmetros: Random forest


Buscando hiperparâmetros: Gradient boosting


Dados reais.


## Partição e validação
Treino no passado; validação e teste em meses posteriores. A validação cruzada prevê cada mês só com os anteriores e nunca toca validação nem teste.

In [2]:
display(R["particao"]); display(R["dobras_cv"])

,conjunto,de,ate,meses,linhas
0,treino,2025-11,2026-05,7,42
1,validação,2026-06,2026-07,2,12
2,teste,2026-08,2026-09,2,12


,dobra,treino_de,treino_ate,valida_em,linhas_treino,linhas_validacao
0,1,2025-11,2025-12,2026-01,12,6
1,2,2025-11,2026-01,2026-02,18,6
2,3,2025-11,2026-02,2026-03,24,6
3,4,2025-11,2026-03,2026-04,30,6
4,5,2025-11,2026-04,2026-05,36,6


## O que cada modelo minimiza e as grades testadas

In [3]:
display(R["grades"])

,modelo,o que minimiza,grade
0,Linear (Ridge),erro quadrático + penalização L2 dos coeficientes,"{""alpha"": [0.01, 0.1, 1.0, 10.0, 100.0]}"
1,Árvore de decisão,erro quadrático (impureza) em cada divisão,"{""max_depth"": [2, 3, 4, 6], ""min_samples_leaf""..."
2,Random forest,erro quadrático por árvore; a média das árvore...,"{""max_depth"": [3, 6, null], ""min_samples_leaf""..."
3,Gradient boosting,"erro quadrático, corrigindo os resíduos das ár...","{""n_estimators"": [100, 300], ""learning_rate"": ..."


## Quadro de comparação (recorte de validação, mesma partição e mesma métrica)
MAE e WAPE por unidade e mês; "erro do total mensal" é o erro absoluto médio da soma das unidades, que é o número que a decisão de capacidade sente.

In [4]:
display(R["comparacao_validacao"].round(3))

,abordagem,tipo,hiperparâmetros,MAE (CV treino),MAE,WAPE,viés,erro do total mensal,custo/tempo
0,Persistência (operacional),baseline,—,NaN,63.833,0.258,-18.500,358.000,desprezível
1,Média do treino,baseline,—,NaN,50.655,0.205,1.274,234.500,desprezível
2,Mediana do treino,baseline,—,NaN,53.083,0.214,-1.083,234.500,desprezível
3,Média móvel de 3 meses,baseline,—,NaN,62.333,0.252,10.889,315.000,desprezível
4,Linear (Ridge),modelo,alpha=0.1,67.324,54.386,0.220,-7.884,252.552,busca 0.3 s; previsão 2.8 ms
5,Árvore de decisão,modelo,"max_depth=4, min_samples_leaf=5",75.160,51.488,0.208,0.976,237.000,busca 0.6 s; previsão 2.6 ms
6,Random forest,modelo,"max_depth=3, max_features=0.5, min_samples_leaf=2",65.017,43.645,0.176,2.371,261.868,busca 6.3 s; previsão 10.0 ms
7,Gradient boosting,modelo,"learning_rate=0.03, max_depth=2, n_estimators=300",76.402,78.703,0.318,-6.191,440.446,busca 2.1 s; previsão 2.5 ms


## Regra de escolha: a complexidade precisa se pagar
O modelo só vence se reduzir o MAE em pelo menos 10% sobre o melhor baseline **e** não piorar o erro do total mensal [H, a combinar com o demandante].

In [5]:
print(R["escolha"])
print(R["ganho_validacao"])

{'melhor_modelo_na_validacao': 'Random forest', 'melhor_baseline_na_validacao': 'Média do treino', 'ganho_relativo_mae': 0.13838957183333767, 'ganho_minimo_exigido': 0.1, 'total_mensal_nao_piora': False, 'recomenda_modelo': False, 'escolhido': 'Média do treino', 'teste_confirma_ganho': True, 'parametros_escolhidos': {'Linear (Ridge)': {'alpha': 0.1}, 'Árvore de decisão': {'max_depth': 4, 'min_samples_leaf': 5}, 'Random forest': {'max_depth': 3, 'max_features': 0.5, 'min_samples_leaf': 2}, 'Gradient boosting': {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 300}}}
{'mae_baseline': 50.654761904761905, 'mae_modelo': 43.64467109344224, 'reducao_relativa_mae': 0.13838957183333767, 'exames_por_unidade_mes_evitados': 7.010090811319664, 'erro_total_mes_baseline': 234.5, 'erro_total_mes_modelo': 261.86802656065333, 'exames_por_mes_no_total_evitados': -27.368026560653334}


## Teste, avaliado uma única vez
Modelos reajustados com treino + validação, mesmos hiperparâmetros. O teste não foi usado para escolher nada.

In [6]:
display(R["comparacao_teste"].round(3))
print(R["ganho_teste"])
print("Teste confirma o ganho da validação?", R["escolha"]["teste_confirma_ganho"])

,abordagem,tipo,MAE,WAPE,viés,erro do total mensal
0,Persistência (operacional),baseline,45.417,0.189,19.417,206.500
1,Média do treino,baseline,38.537,0.160,8.241,49.444
2,Mediana do treino,baseline,46.500,0.193,5.167,45.000
3,Média móvel de 3 meses,baseline,28.528,0.119,5.139,61.833
4,Linear (Ridge),modelo,38.964,0.162,12.171,73.028
5,Árvore de decisão,modelo,38.516,0.160,8.725,52.350
6,Random forest,modelo,27.494,0.114,6.894,41.367
7,Gradient boosting,modelo,36.279,0.151,3.430,45.804


{'mae_baseline': 38.53703703703704, 'mae_modelo': 27.494435485887426, 'reducao_relativa_mae': 0.28654516278812064, 'exames_por_unidade_mes_evitados': 11.042601551149613, 'erro_total_mes_baseline': 49.44444444444457, 'erro_total_mes_modelo': 41.366528981863325, 'exames_por_mes_no_total_evitados': 8.077915462581245}
Teste confirma o ganho da validação? True
